# Assignment 2: Stock Market Analysis Agent

This notebook demonstrates a LangGraph-based stock market analysis agent built from scratch.

The agent accepts a stock ticker symbol, retrieves historical stock data, calculates technical indicators, generates a BUY/HOLD/SELL recommendation, and displays a formatted analysis report.

The implementation focuses on:

- LangGraph workflow design
- state propagation between nodes
- technical indicator calculation
- recommendation logic
- error handling
- working demonstrations
- unit testing

## Agent Architecture

The stock analysis agent follows a sequential LangGraph workflow:

```text
START
  ↓
validate_ticker
  ↓
fetch_stock_data
  ↓
calculate_indicators
  ↓
generate_recommendation
  ↓
format_report
  ↓
END
```

Each node has one responsibility. This makes the agent easier to test, debug, and maintain.

## State Design

LangGraph passes a shared state object from node to node.

The stock agent state contains:

| Field | Purpose |
|---|---|
| `ticker` | User-provided stock ticker symbol |
| `stock_data` | Historical stock data fetched from yfinance |
| `indicators` | Calculated SMA and RSI values |
| `recommendation` | Final BUY/HOLD/SELL recommendation |
| `report` | Formatted final report |
| `error` | Optional error message |

This structure allows each node to read required values and add new results for downstream nodes.

## Environment Setup

The project dependencies are listed in `assignment_2/stock_agent/requirements.txt`.

If dependencies are not yet installed, run this from the `assignment_2/stock_agent` folder:

```bash
pip install -r requirements.txt
```

The notebook assumes it is being run from the repository root or from an environment where the `assignment_2/stock_agent` folder is accessible.

In [1]:
# Optional dependency installation cell.
# Uncomment and run only if the packages are not installed yet.
%pip install yfinance pandas numpy langgraph langchain langchain-core pydantic pydantic-settings
#%pip install -r assignment_2/stock_agent/requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Import Setup

The stock agent code is stored under `assignment_2/stock_agent`. This cell adds that folder to the Python path so the notebook can import the graph and components.

In [2]:
import sys
from pathlib import Path

stock_agent_path = Path.cwd() / "assignment_2" / "stock_agent"

if not stock_agent_path.exists():
    stock_agent_path = Path.cwd() / "stock_agent"

if not stock_agent_path.exists():
    stock_agent_path = Path.cwd()

sys.path.insert(0, str(stock_agent_path))

print(f"Using stock agent path: {stock_agent_path}")

Using stock agent path: d:\Documents\Springer Capital Files\weather_agent\weather-agent-debugging\assignment_2\stock_agent


## Technical Indicator Logic

The agent calculates three required technical indicators:

### Simple Moving Average

The Simple Moving Average smooths price data by averaging the closing price over a fixed window.

- 10-day SMA reflects shorter-term price movement
- 20-day SMA reflects slightly longer-term price movement

### Relative Strength Index

The RSI measures recent price momentum. It compares average gains and average losses over a specified period, usually 14 days.

Common interpretation:

- RSI below 30 may indicate oversold conditions
- RSI above 70 may indicate overbought conditions
- RSI between 30 and 70 is usually neutral/moderate

In [3]:
from components.indicators import calculate_sma, calculate_rsi
import pandas as pd

sample_prices = pd.DataFrame({
    "Close": [100, 101, 102, 103, 104, 105, 106, 107, 108, 109,
              110, 111, 112, 113, 114, 115, 116, 117, 118, 119]
})

sample_prices["SMA_10"] = calculate_sma(sample_prices, 10)
sample_prices["RSI_14"] = calculate_rsi(sample_prices, 14)

sample_prices.tail()

,Close,SMA_10,RSI_14
15,115,110.5,100.0
16,116,111.5,100.0
17,117,112.5,100.0
18,118,113.5,100.0
19,119,114.5,100.0


## Recommendation Logic

The recommendation logic is intentionally simple and rule-based.

### BUY Condition

The agent recommends BUY when:

```text
current price > SMA 10 > SMA 20
and RSI < 70
```

This suggests that the stock has upward momentum while not yet being extremely overbought.

### SELL Condition

The agent recommends SELL when:

```text
current price < SMA 10 < SMA 20
and RSI > 30
```

This suggests weakening price action while not yet being extremely oversold.

### HOLD Condition

If neither BUY nor SELL conditions are met, the agent recommends HOLD.

This avoids forcing a strong recommendation when the indicators are mixed or unclear.

## Import and Compile the LangGraph Agent

This imports the compiled LangGraph application from `graph.py`.

The graph must be compiled using `builder.compile()` before it can be executed with `.invoke()`.

In [4]:
from graph import stock_agent

print("Stock agent imported successfully.")
print(type(stock_agent))

Stock agent imported successfully.
<class 'langgraph.graph.state.CompiledStateGraph'>


## Working Demonstration Using Live yfinance Data

This cell runs the full agent using a real stock ticker.

Because this uses `yfinance`, it requires internet access. If the network is unavailable, the mocked demonstration below still verifies the agent's logic.

In [5]:
initial_state = {
    "ticker": "AAPL",
    "stock_data": None,
    "indicators": None,
    "recommendation": None,
    "report": None,
    "error": None,
}

try:
    final_state = stock_agent.invoke(initial_state)
    print(final_state["report"])
except Exception as error:
    print("Live yfinance demo failed. This may be due to network or API availability.")
    print(error)

Stock Market Analysis Report

Ticker Symbol: AAPL

Technical Indicators:
- Current Price: $287.44
- SMA (10-Day): $276.7
- SMA (20-Day): $271.57
- RSI (14-Day): 67.06

Recommendation:
>>> BUY



## Mocked Working Demonstration

A mocked demonstration is useful because it does not depend on internet access or the availability of external financial APIs.

This proves that the LangGraph workflow, technical indicator calculations, recommendation logic, and report formatting work correctly under controlled data.

In [6]:
from unittest.mock import patch
import pandas as pd


def create_mock_stock_data():
    dates = pd.date_range(start="2026-01-01", periods=30, freq="D")
    close_prices = [
        100, 101, 102, 103, 104,
        105, 106, 107, 108, 109,
        110, 111, 112, 113, 114,
        115, 116, 117, 118, 119,
        120, 121, 122, 123, 124,
        125, 126, 127, 128, 129,
    ]

    return pd.DataFrame(
        {
            "Open": close_prices,
            "High": close_prices,
            "Low": close_prices,
            "Close": close_prices,
            "Volume": [1000000] * 30,
        },
        index=dates,
    )


class MockTicker:
    def __init__(self, ticker):
        self.ticker = ticker

    def history(self, period="60d", interval="1d"):
        return create_mock_stock_data()


with patch("components.nodes.yf.Ticker", MockTicker):
    mocked_state = stock_agent.invoke(
        {
            "ticker": "MOCK",
            "stock_data": None,
            "indicators": None,
            "recommendation": None,
            "report": None,
            "error": None,
        }
    )

print(mocked_state["report"])

Stock Market Analysis Report

Ticker Symbol: MOCK

Technical Indicators:
- Current Price: $129.0
- SMA (10-Day): $124.5
- SMA (20-Day): $119.5
- RSI (14-Day): 100.0

Recommendation:
>>> HOLD



## Error Handling Demonstration

The agent should handle invalid or unavailable tickers gracefully.

In this demonstration, the mocked ticker returns an empty DataFrame, simulating an invalid ticker or unavailable stock data.

In [7]:
class EmptyMockTicker:
    def __init__(self, ticker):
        self.ticker = ticker

    def history(self, period="60d", interval="1d"):
        return pd.DataFrame()


with patch("components.nodes.yf.Ticker", EmptyMockTicker):
    try:
        stock_agent.invoke(
            {
                "ticker": "INVALIDTICKER",
                "stock_data": None,
                "indicators": None,
                "recommendation": None,
                "report": None,
                "error": None,
            }
        )
    except Exception as error:
        print("Handled error successfully:")
        print(error)

Handled error successfully:
Failed to fetch stock data: No stock data found for ticker: INVALIDTICKER


## Unit Testing

The test suite validates both individual calculations and full LangGraph execution.

The tests cover:

- SMA calculation
- RSI calculation
- successful report generation
- invalid ticker handling

Mocked stock data is used so the tests remain deterministic and do not depend on external API availability.

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "unittest", "test_stock_agent.py"],
    cwd=stock_agent_path,
    capture_output=True,
    text=True,
)

print(result.stdout)
print(result.stderr)    


....
----------------------------------------------------------------------
Ran 4 tests in 0.119s

OK



## Final Result

The completed Assignment 2 implementation demonstrates:

- a LangGraph agent built from scratch
- clean node-based architecture
- technical indicator calculations using pandas
- BUY/HOLD/SELL recommendation logic
- formatted stock analysis reports
- graceful handling of invalid ticker scenarios
- deterministic testing with mocked stock data

This makes the stock analysis agent reproducible, testable, and easier to maintain.

## Reflection

This assignment strengthened my understanding of how LangGraph can be used to build structured, multi-step agents.

Unlike Assignment 1, which focused on debugging an existing workflow, this assignment required designing the agent architecture from scratch.

The most important design decision was separating the workflow into clear nodes:

- input validation
- data fetching
- indicator calculation
- recommendation generation
- report formatting

This separation makes the system easier to reason about, test, and extend in the future.